# Stage 3 InternVL3 2B Clean Triage

Setup runs first, then inference runs in a fresh Python process to avoid Kaggle package-state conflicts.

In [ ]:

from pathlib import Path
import subprocess

WORK_DIR = Path('/kaggle/working')
RUNNER_PATH = WORK_DIR / 'stage3_vlm_triage_internvl3_2b_clean_runner.py'
RUNNER_CODE = r"""
import json
import re
import shutil
import subprocess
import tarfile
import time
import traceback
from pathlib import Path

MODEL_KEY = "internvl3_2b_hf"
MODEL_ID = "OpenGVLab/InternVL3-2B-hf"
MODEL_FAMILY = "internvl"
RUN_ID = "stage3_vlm_triage_" + MODEL_KEY + "_clean"
REPO_URL = "https://github.com/konstRyaz/vlm-for-insulator-defect-detection.git"
REPO_DIR = Path("/tmp/vlm-for-insulator-defect-detection")
WORK_DIR = Path("/kaggle/working")
RUN_DIR = WORK_DIR / RUN_ID
DATASET_ROOT_CANDIDATES = [Path("/kaggle/input/datasets/kostyaryazanov/idid-coco-v3"), Path("/kaggle/input/idid-coco-v3")]
JSONL_REL = Path("stage3_regrouped_v2/val/vlm_labels_v1_val_v2.annotated.jsonl")
PROMPT_VERSION = "qwen_vlm_labels_v1_prompt_v7f_flashover_unclear_to_unknown_nocroppath"
MAX_NEW_TOKENS = 220


def write_json(path, obj):
    path.parent.mkdir(parents=True, exist_ok=True)
    path.write_text(json.dumps(obj, indent=2, ensure_ascii=False), encoding="utf-8")


def write_jsonl(path, rows):
    path.parent.mkdir(parents=True, exist_ok=True)
    with path.open("w", encoding="utf-8") as f:
        for row in rows:
            f.write(json.dumps(row, ensure_ascii=False) + "\n")


def pack_run_dir():
    archive = WORK_DIR / f"stage3_deliverables_{RUN_ID}.tar.gz"
    if archive.exists():
        archive.unlink()
    with tarfile.open(archive, "w:gz") as tar:
        if RUN_DIR.exists():
            tar.add(RUN_DIR, arcname=RUN_ID)
    print("Archive:", archive)


def sh(args, cwd=None, check=True):
    print("$", " ".join(str(a) for a in args))
    proc = subprocess.run([str(a) for a in args], cwd=str(cwd) if cwd else None, text=True, capture_output=True)
    if proc.stdout:
        print(proc.stdout[-4000:])
    if proc.stderr:
        print(proc.stderr[-4000:])
    if check and proc.returncode != 0:
        raise RuntimeError(f"command failed ({proc.returncode}): {' '.join(str(a) for a in args)}")
    return proc


def extract_json_object(text):
    if not isinstance(text, str):
        return None, "not a string"
    cleaned = text.strip()
    cleaned = re.sub(r"^```(?:json)?", "", cleaned, flags=re.IGNORECASE).strip()
    cleaned = re.sub(r"```$", "", cleaned).strip()
    start = cleaned.find("{")
    end = cleaned.rfind("}")
    if start < 0 or end < start:
        return None, "no JSON object found"
    try:
        obj = json.loads(cleaned[start:end + 1])
    except Exception as exc:
        return None, str(exc)
    return obj if isinstance(obj, dict) else None, "" if isinstance(obj, dict) else "JSON root is not an object"

ALLOWED_CLASSES = {"insulator_ok", "defect_flashover", "defect_broken", "unknown", "other"}
ALLOWED_VIS = {"clear", "partial", "ambiguous"}


def normalize_prediction(obj):
    errors = []
    if not isinstance(obj, dict):
        obj = {}
        errors.append("missing object")
    coarse = obj.get("coarse_class", "unknown")
    if coarse not in ALLOWED_CLASSES:
        errors.append(f"invalid coarse_class: {coarse}")
        coarse = "unknown"
    vis = obj.get("visibility", "ambiguous")
    if vis not in ALLOWED_VIS:
        errors.append(f"invalid visibility: {vis}")
        vis = "ambiguous"
    tags = obj.get("visual_evidence_tags", [])
    if not isinstance(tags, list):
        errors.append("visual_evidence_tags is not a list")
        tags = []
    tags = [str(t).strip() for t in tags if str(t).strip()][:8]
    desc = obj.get("short_canonical_description_en") or obj.get("short_canonical_description") or "Visual evidence is inconclusive."
    rep = obj.get("report_snippet_en") or obj.get("report_snippet") or "The crop does not provide enough reliable evidence for a more specific report."
    out = {
        "coarse_class": coarse,
        "visual_evidence_tags": tags,
        "visibility": vis,
        "short_canonical_description_en": str(desc).strip()[:240] or "Visual evidence is inconclusive.",
        "report_snippet_en": str(rep).strip()[:360] or "The crop does not provide enough reliable evidence for a more specific report.",
    }
    if "annotator_notes" in obj:
        out["annotator_notes"] = str(obj.get("annotator_notes", ""))[:240]
    return out, errors


def to_vlm_labels_record(gt_row, normalized):
    rec = {k: gt_row.get(k) for k in ["record_id", "image_id", "box_id", "source", "split", "bbox_xywh", "crop_path", "image_path", "score", "category_name", "label_version"]}
    rec.update({
        "coarse_class": normalized["coarse_class"],
        "visual_evidence_tags": normalized["visual_evidence_tags"],
        "visibility": normalized["visibility"],
        "needs_review": normalized["visibility"] == "ambiguous",
        "short_canonical_description": normalized["short_canonical_description_en"],
        "short_canonical_description_en": normalized["short_canonical_description_en"],
        "report_snippet": normalized["report_snippet_en"],
        "report_snippet_en": normalized["report_snippet_en"],
        "annotator_notes": normalized.get("annotator_notes", ""),
    })
    return rec


def normalize_generated_text(result):
    item = result[0] if isinstance(result, list) and result else result
    gen = item.get("generated_text", item.get("text", "")) if isinstance(item, dict) else item
    if isinstance(gen, list):
        for msg in reversed(gen):
            if isinstance(msg, dict) and msg.get("role") == "assistant":
                content = msg.get("content", "")
                if isinstance(content, list):
                    return "\n".join(str(x.get("text", x)) if isinstance(x, dict) else str(x) for x in content)
                return str(content)
        return str(gen[-1]) if gen else ""
    return str(gen)


def main():
    RUN_DIR.mkdir(parents=True, exist_ok=True)
    write_json(RUN_DIR / "job_info.json", {"model_key": MODEL_KEY, "model_id": MODEL_ID, "run_id": RUN_ID})
    if sh(["nvidia-smi"], check=False).returncode != 0:
        raise RuntimeError("GPU is required for this run")

    data_root = next((c for c in DATASET_ROOT_CANDIDATES if (c / JSONL_REL).exists()), None)
    if data_root is None:
        raise FileNotFoundError("clean Stage 3 validation JSONL was not found")
    input_jsonl = data_root / JSONL_REL

    if REPO_DIR.exists():
        shutil.rmtree(REPO_DIR)
    sh(["git", "clone", "--depth", "1", REPO_URL, str(REPO_DIR)])

    import pandas as pd
    import torch
    import yaml
    from PIL import Image
    from transformers import pipeline

    base_cfg = yaml.safe_load((REPO_DIR / "configs/pipeline/stage3_vlm_gt_baseline.yaml").read_text(encoding="utf-8"))
    prompt_entry = base_cfg["prompt"]["versions"][PROMPT_VERSION]
    system_prompt = (REPO_DIR / prompt_entry["system_path"]).read_text(encoding="utf-8")
    user_template = (REPO_DIR / prompt_entry["user_path"]).read_text(encoding="utf-8")
    if "crop_path" in user_template:
        raise RuntimeError("selected prompt exposes crop_path")
    records = [json.loads(line) for line in input_jsonl.read_text(encoding="utf-8").splitlines() if line.strip()]
    print("records:", len(records))

    def resolve_crop(row):
        rel = Path(row["crop_path"])
        for p in [input_jsonl.parent / rel, data_root / "stage3_regrouped_v2" / "val" / rel, data_root / "stage3_regrouped_v2" / rel]:
            if p.exists():
                return p
        raise FileNotFoundError(row["crop_path"])

    def build_user_prompt(row):
        text = user_template
        for key, value in {"record_id": row.get("record_id", ""), "split": row.get("split", ""), "source": row.get("source", ""), "bbox_xywh": json.dumps(row.get("bbox_xywh", []), ensure_ascii=False)}.items():
            text = text.replace("{{" + key + "}}", str(value))
        return text

    kwargs = {"model": MODEL_ID, "trust_remote_code": True, "device_map": "auto"}
    kwargs["torch_dtype"] = torch.float16
    kwargs["attn_implementation"] = "eager"
    print("torch", torch.__version__, "cuda", torch.version.cuda, "available", torch.cuda.is_available())
    if torch.cuda.is_available():
        print("device", torch.cuda.get_device_name(0))
        _ = (torch.ones(1, device="cuda") + 1).cpu()
    print("Loading", MODEL_ID)
    pipe = pipeline("image-text-to-text", **kwargs)
    print("Model loaded")

    def generate_one(image, prompt_text):
        messages = [{"role": "user", "content": [{"type": "image", "image": image}, {"type": "text", "text": system_prompt + "\n\n" + prompt_text}]}]
        result = pipe(text=messages, max_new_tokens=MAX_NEW_TOKENS, do_sample=False, return_full_text=False)
        return normalize_generated_text(result)

    raw_rows, parsed_rows, pred_rows, sample_rows = [], [], [], []
    start_time = time.time()
    for idx, row in enumerate(records, start=1):
        rid = row["record_id"]
        t0 = time.time()
        raw_text = ""
        parsed_obj = None
        parse_error = ""
        parse_status = "failed"
        schema_valid = False
        schema_errors = []
        status = "error"
        try:
            raw_text = generate_one(Image.open(resolve_crop(row)).convert("RGB"), build_user_prompt(row))
            parsed_obj, parse_error = extract_json_object(raw_text)
            if parsed_obj is not None:
                parse_status = "success"
            normalized, schema_errors = normalize_prediction(parsed_obj)
            schema_valid = len(schema_errors) == 0
            status = "ok"
        except Exception as exc:
            parse_error = repr(exc)
            normalized, schema_errors = normalize_prediction(None)
        pred_rows.append(to_vlm_labels_record(row, normalized))
        raw_rows.append({"record_id": rid, "raw_response": raw_text})
        parsed_rows.append({"record_id": rid, "raw_prediction": parsed_obj, "normalized_prediction": normalized, "normalization_errors": schema_errors})
        sample_rows.append({"record_id": rid, "status": status, "parse_status": parse_status, "parse_error": parse_error, "schema_valid": schema_valid, "schema_errors": schema_errors, "elapsed_sec": round(time.time() - t0, 3)})
        if idx == 1 or idx % 5 == 0:
            print(f"{idx}/{len(records)} {rid} {status} {parse_status} schema={schema_valid} elapsed={time.time()-t0:.1f}s")

    write_jsonl(RUN_DIR / "raw_responses.jsonl", raw_rows)
    write_jsonl(RUN_DIR / "parsed_predictions.jsonl", parsed_rows)
    write_jsonl(RUN_DIR / "predictions_vlm_labels_v1.jsonl", pred_rows)
    write_jsonl(RUN_DIR / "sample_results.jsonl", sample_rows)
    write_json(RUN_DIR / "run_summary.json", {"model_key": MODEL_KEY, "model_id": MODEL_ID, "run_id": RUN_ID, "prompt_version": PROMPT_VERSION, "total": len(records), "parse_success": sum(1 for r in sample_rows if r["parse_status"] == "success"), "schema_valid": sum(1 for r in sample_rows if r["schema_valid"]), "elapsed_sec": round(time.time() - start_time, 3)})

    sh(["python", "scripts/validate_vlm_labels_v1.py", "--input", str(RUN_DIR / "predictions_vlm_labels_v1.jsonl")], cwd=REPO_DIR, check=False)
    sh(["python", "scripts/eval_stage3_vlm_baseline.py", "--run-dir", str(RUN_DIR), "--ground-truth-jsonl", str(input_jsonl), "--output-dir", str(RUN_DIR / "eval")], cwd=REPO_DIR, check=False)
    metrics_path = RUN_DIR / "eval" / "metrics.json"
    if metrics_path.exists():
        metrics = json.loads(metrics_path.read_text(encoding="utf-8"))
        rates = metrics.get("rates", {})
        row = {"model_key": MODEL_KEY, "model_id": MODEL_ID, "run_id": RUN_ID, "evaluated_total": metrics.get("counts", {}).get("evaluated_total"), "parse_success": rates.get("parse_success_rate"), "schema_valid": rates.get("schema_valid_rate"), "coarse_acc": rates.get("coarse_accuracy"), "coarse_macro_f1": metrics.get("coarse_class", {}).get("macro_f1"), "visibility_acc": rates.get("visibility_accuracy"), "visibility_macro_f1": metrics.get("visibility", {}).get("macro_f1"), "tag_mean_jaccard": rates.get("tag_mean_jaccard")}
        pd.DataFrame([row]).to_csv(RUN_DIR / "comparison_row.csv", index=False)
        print(pd.DataFrame([row]).to_markdown(index=False))
    pack_run_dir()

try:
    main()
except Exception:
    RUN_DIR.mkdir(parents=True, exist_ok=True)
    report = traceback.format_exc()
    print(report)
    (RUN_DIR / "error_report.txt").write_text(report, encoding="utf-8")
    write_json(RUN_DIR / "run_summary.json", {"model_key": MODEL_KEY, "model_id": MODEL_ID, "run_id": RUN_ID, "error": report})
    pack_run_dir()
    raise
"""
RUNNER_PATH.write_text(RUNNER_CODE, encoding='utf-8')


def sh(args, check=True):
    print('$', ' '.join(str(a) for a in args))
    proc = subprocess.run([str(a) for a in args], text=True, capture_output=True)
    if proc.stdout:
        print(proc.stdout[-4000:])
    if proc.stderr:
        print(proc.stderr[-4000:])
    if check and proc.returncode != 0:
        raise RuntimeError(f'command failed ({proc.returncode})')
    return proc

sh(['python', '-m', 'pip', 'install', '-q', '--force-reinstall', '--no-cache-dir', 'torch==2.5.1', 'torchvision==0.20.1', '--index-url', 'https://download.pytorch.org/whl/cu121'])
sh(['python', '-m', 'pip', 'install', '-q', '--force-reinstall', '--no-cache-dir', 'Pillow==11.3.0'])
sh(['python', '-m', 'pip', 'install', '-q', '-U', 'transformers>=4.57.0', 'accelerate', 'pandas', 'pyyaml', 'tabulate', 'sentencepiece', 'protobuf', 'timm', 'einops'])
sh(['python', str(RUNNER_PATH)])
